# chrome

> Shared chrome switching route handlers for the combined Phase 2 step

In [ ]:
#| default_exp routes.core.chrome

In [ ]:
#| export
from typing import Tuple, Dict, Callable

from fasthtml.common import APIRouter, Div

from cjm_fasthtml_card_stack.components.controls import render_width_slider
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_fasthtml_workflow_transcript_decomp.core.html_ids import StructureDecompHtmlIds
from cjm_fasthtml_workflow_transcript_decomp.core.models import WorkingSegment, VADChunk
from cjm_fasthtml_workflow_transcript_decomp.routes.models import DecompUrls, AlignmentUrls
from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

# Decomposition renderers
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.step_renderer import (
    render_toolbar as render_decomp_toolbar,
    render_decomp_footer_content,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.card_stack_config import (
    DECOMP_CS_CONFIG, DECOMP_CS_IDS,
)

# Alignment renderers
from cjm_fasthtml_workflow_transcript_decomp.components.step_alignment.step_renderer import (
    render_align_toolbar, render_align_footer_content,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_alignment.card_stack_config import (
    ALIGN_CS_CONFIG, ALIGN_CS_IDS,
)

# Footer helper
from cjm_fasthtml_workflow_transcript_decomp.components.step_combined import (
    render_footer_inner_content,
)

# Keyboard config
from cjm_fasthtml_workflow_transcript_decomp.components.keyboard_config import (
    build_combined_kb_system, render_keyboard_hints_collapsible,
)

DEBUG_SWITCH_CHROME = False

## Handlers

In [ ]:
#| export
async def _handle_switch_chrome(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    decomp_urls:DecompUrls,  # URL bundle for decomposition routes
    align_urls:AlignmentUrls,  # URL bundle for alignment routes
) -> tuple:  # OOB swaps for shared chrome containers
    """Switch shared chrome content based on active column."""
    form = await request.form()
    active_column = form.get("active_column", "decomp")

    if DEBUG_SWITCH_CHROME:
        print(f"[SWITCH_CHROME] active_column: {active_column}")

    session_id = get_session_id(sess)
    state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    step_states = state.get("step_states", {})
    decomp_state = step_states.get("decomposition", {})
    align_state = step_states.get("alignment", {})

    # Get counts for alignment status
    segment_count = len(decomp_state.get("segments", []))
    chunk_count = len(align_state.get("vad_chunks", []))

    # Build combined KB manager for hints
    kb_manager, _ = build_combined_kb_system(decomp_urls, align_urls)

    if active_column == "decomp":
        # Decomp chrome
        segments = [WorkingSegment.from_dict(s) for s in decomp_state.get("segments", [])]
        history = decomp_state.get("history", [])
        focused_index = decomp_state.get("focused_index", 0)
        visible_count = decomp_state.get("visible_count", DEFAULT_VISIBLE_COUNT)
        is_auto_mode = decomp_state.get("is_auto_mode", False)
        card_width = decomp_state.get("card_width", DEFAULT_CARD_WIDTH)

        hints_content = render_keyboard_hints_collapsible(kb_manager, include_zone_switch=True)
        toolbar_content = render_decomp_toolbar(
            reset_url=decomp_urls.reset,
            ai_split_url=decomp_urls.ai_split,
            undo_url=decomp_urls.undo,
            can_undo=(len(history) > 0),
            visible_count=visible_count,
            is_auto_mode=is_auto_mode,
        )
        controls_content = render_width_slider(DECOMP_CS_CONFIG, DECOMP_CS_IDS, card_width=card_width)
        column_footer = render_decomp_footer_content(segments, focused_index)
    else:
        # Alignment chrome
        chunks = [VADChunk.from_dict(c) for c in align_state.get("vad_chunks", [])]
        focused_index = align_state.get("focused_chunk_index", 0)
        visible_count = align_state.get("visible_count", 5)
        is_auto_mode = align_state.get("is_auto_mode", False)
        card_width = align_state.get("card_width", 40)

        hints_content = render_keyboard_hints_collapsible(kb_manager, include_zone_switch=True)
        toolbar_content = render_align_toolbar(
            visible_count=visible_count,
            is_auto_mode=is_auto_mode,
        )
        controls_content = render_width_slider(ALIGN_CS_CONFIG, ALIGN_CS_IDS, card_width=card_width)
        column_footer = render_align_footer_content(chunks, focused_index)

    if DEBUG_SWITCH_CHROME:
        print(f"[SWITCH_CHROME] returning OOB swaps for {active_column}")

    # Return OOB swaps
    hints_oob = Div(
        hints_content,
        id=StructureDecompHtmlIds.SHARED_HINTS,
        hx_swap_oob="innerHTML"
    )
    toolbar_oob = Div(
        toolbar_content,
        id=StructureDecompHtmlIds.SHARED_TOOLBAR,
        hx_swap_oob="innerHTML"
    )
    controls_oob = Div(
        controls_content,
        id=StructureDecompHtmlIds.SHARED_CONTROLS,
        hx_swap_oob="innerHTML"
    )
    footer_oob = Div(
        render_footer_inner_content(column_footer, segment_count, chunk_count),
        id=StructureDecompHtmlIds.SHARED_FOOTER,
        hx_swap_oob="innerHTML"
    )

    return (hints_oob, toolbar_oob, controls_oob, footer_oob)

## Router

In [ ]:
#| export
def init_chrome_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/core/chrome")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize chrome switching routes."""
    router = APIRouter(prefix=prefix)

    @router
    async def switch_chrome(request, sess):
        """Switch shared chrome content based on active column."""
        return await _handle_switch_chrome(
            workflow, request, sess,
            decomp_urls=workflow._decomp_urls,
            align_urls=workflow._align_urls,
        )

    routes = {
        "switch_chrome": switch_chrome,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()